## Pre-Build Middleware
- PII
- HumanInTheLoop
- ModelFallback

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)
from langchain.chat_models import init_chat_model
import os

qwen_model = init_chat_model("openai:qwen3.7-plus", temperature=0, base_url=os.getenv("ROOT_URL"))


In [4]:
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

# =================1.定义PIIMiddleware==========================
# 也可以添加多个PII中间件来检测不同类型的敏感信息
pii_middleware_email = PIIMiddleware(
    "email",
    strategy="redact",  # mask
    apply_to_input=False,
    apply_to_output=True
)

pii_middleware_phone = PIIMiddleware(
    "phone_number",
    # 使用自定义正则表达式
    detector=r"(?:\+?\d{1,3}[\s.-]?)?(?:\(?\d{2,4}\)?[\s.-]?)?\d{3,4}[\s.-]?\d{4}",
    strategy="block",
    apply_to_input=True
)

# ====================2.在Agent中注册PIIMiddleware================
agent_with_builtin_middleware = create_agent(
    model=qwen_model,
    middleware=[pii_middleware_email, pii_middleware_phone]
)

# ======================3.测试=================================
response = agent_with_builtin_middleware.invoke({
    "messages": [
        HumanMessage(
            "我的邮箱是: huge@itcast.cn,我的电话是13698023405,你要记住这些信息，后续要用到。如果记住了请确认一遍。")
    ]
})
for m in response['messages']:
    m.pretty_print()

PIIDetectionError: Detected 1 instance(s) of phone_number in text content

In [10]:
from langchain.agents.middleware import ModelFallbackMiddleware

fake_primary_model = init_chat_model("openai:fake-primary-model", temperature=0, base_url=os.getenv("ROOT_URL"))

model_fallback_middleware = ModelFallbackMiddleware(
   fake_primary_model,  # 默认模型
    qwen_model  # 备用模型
)
agent_with_model_fallback = create_agent(
    model=fake_primary_model,
    middleware=[model_fallback_middleware]
)

# 3.测试
response = agent_with_builtin_middleware.invoke({
    "messages": [HumanMessage("你好")]
})
for m in response['messages']:
    m.pretty_print()

================================ Human Message =================================

你好
================================== Ai Message ==================================

你好！有什么我可以帮你的吗？


In [14]:

emails = [
        {
            "subject": "周末见个面？",
            "content": """
                嗨 虎哥，
                我下周会去城里，不知道我们有没有机会一起喝杯咖啡？

                祝好，简
            """,
            "from": "jane@itcast.cn",
            "status": "unread"
        },
        {
            "subject": "周五会议",
            "content": """
                嗨 虎哥，
                非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？

                祝好，小李
            """,
            "from": "lixiaolong@itcast.cn",
            "status": "checked"
        }
    ]


from langchain.tools import tool

# 定义工具，收取和发送邮件
@tool
def check_inbox() -> str:
    """查看收件箱中的邮件."""

    # 模拟收件箱邮件
    return emails


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """发送回复邮件"""
    return f"邮件已发送至 {to} , 主题： {subject} , 内容： {body}"


from langchain.agents.middleware import HumanInTheLoopMiddleware

# 创建人工审核中间件
human_in_loop_middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        "send_email": {
            "description": "请确认邮件内容",
            "allowed_decisions": ["approve", "reject", "edit"]
        }
    }
)

from langgraph.checkpoint.memory import InMemorySaver
agent_with_hitl = create_agent(
    model=qwen_model,
    tools=[check_inbox, send_email],
    middleware=[human_in_loop_middleware],
    checkpointer=InMemorySaver(),
    system_prompt="你是一个邮件助手"
)

config = {"configurable": {"thread_id": "1"}}
response = agent_with_hitl.invoke(
    {"messages": [HumanMessage("帮我查看邮件")]},
    config=config
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

帮我查看邮件
================================== Ai Message ==================================
Tool Calls:
  check_inbox (call_bc4a575d129a423f891bdf73)
 Call ID: call_bc4a575d129a423f891bdf73
  Args:
================================= Tool Message =================================
Name: check_inbox

[{"subject": "周末见个面？", "content": "\n                嗨 虎哥，\n                我下周会去城里，不知道我们有没有机会一起喝杯咖啡？\n\n                祝好，简\n            ", "from": "jane@itcast.cn", "status": "unread"}, {"subject": "周五会议", "content": "\n                嗨 虎哥，\n                非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？\n\n                祝好，小李\n            ", "from": "lixiaolong@itcast.cn", "status": "checked"}]
================================== Ai Message ==================================

您的收件箱中有 2 封邮件：

**1. 周末见个面？** (未读)
- 发件人：jane@itcast.cn
- 内容：简邀请您下周一起喝咖啡

**2. 周五会议** (已查看)
- 发件人：lixiaolong@itcast.cn
- 内容：小李无法准时参加周五的会议，请求重新安排时间

需要我帮您回

In [15]:
response = agent_with_hitl.invoke(
    {"messages": [HumanMessage("帮我回复Jane，很高兴她能来，问问她具体见面时间")]},
    config=config # 注意使用同一个config，也就是thread_id
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

帮我查看邮件
================================== Ai Message ==================================
Tool Calls:
  check_inbox (call_bc4a575d129a423f891bdf73)
 Call ID: call_bc4a575d129a423f891bdf73
  Args:
================================= Tool Message =================================
Name: check_inbox

[{"subject": "周末见个面？", "content": "\n                嗨 虎哥，\n                我下周会去城里，不知道我们有没有机会一起喝杯咖啡？\n\n                祝好，简\n            ", "from": "jane@itcast.cn", "status": "unread"}, {"subject": "周五会议", "content": "\n                嗨 虎哥，\n                非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？\n\n                祝好，小李\n            ", "from": "lixiaolong@itcast.cn", "status": "checked"}]
================================== Ai Message ==================================

您的收件箱中有 2 封邮件：

**1. 周末见个面？** (未读)
- 发件人：jane@itcast.cn
- 内容：简邀请您下周一起喝咖啡

**2. 周五会议** (已查看)
- 发件人：lixiaolong@itcast.cn
- 内容：小李无法准时参加周五的会议，请求重新安排时间

需要我帮您回

In [16]:
print(response)

{'messages': [HumanMessage(content='帮我查看邮件', additional_kwargs={}, response_metadata={}, id='cadaedc4-cab5-4486-9378-f9ed10fce13f'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 338, 'total_tokens': 385, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 29, 'rejected_prediction_tokens': None, 'text_tokens': 47}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'text_tokens': 338}}, 'model_provider': 'openai', 'model_name': 'qwen3.7-plus', 'system_fingerprint': None, 'id': 'chatcmpl-e10ed593-79e7-95c2-b9b5-4b86dc37302f', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1006c-0f0d-7e40-8944-ab03fa2a082b-0', tool_calls=[{'name': 'check_inbox', 'args': {}, 'id': 'call_bc4a575d129a423f891bdf73', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 338, 'outp

In [17]:
from langgraph.types import Command

response = agent_with_hitl.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject",            # 人工确认的决定，是reject
                    "message": "语气太书面了，Jane是我的女朋友！" # 请求被拒绝的原因
                }
            ]
        }
    ),
    config=config  # 相同的thread_id以恢复暂停的对话
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

帮我查看邮件
================================== Ai Message ==================================
Tool Calls:
  check_inbox (call_bc4a575d129a423f891bdf73)
 Call ID: call_bc4a575d129a423f891bdf73
  Args:
================================= Tool Message =================================
Name: check_inbox

[{"subject": "周末见个面？", "content": "\n                嗨 虎哥，\n                我下周会去城里，不知道我们有没有机会一起喝杯咖啡？\n\n                祝好，简\n            ", "from": "jane@itcast.cn", "status": "unread"}, {"subject": "周五会议", "content": "\n                嗨 虎哥，\n                非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？\n\n                祝好，小李\n            ", "from": "lixiaolong@itcast.cn", "status": "checked"}]
================================== Ai Message ==================================

您的收件箱中有 2 封邮件：

**1. 周末见个面？** (未读)
- 发件人：jane@itcast.cn
- 内容：简邀请您下周一起喝咖啡

**2. 周五会议** (已查看)
- 发件人：lixiaolong@itcast.cn
- 内容：小李无法准时参加周五的会议，请求重新安排时间

需要我帮您回

In [18]:
from langgraph.types import Command

response = agent_with_hitl.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "approve",            # 人工确认的决定，是reject
                }
            ]
        }
    ),
    config=config  # 相同的thread_id以恢复暂停的对话
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

帮我查看邮件
================================== Ai Message ==================================
Tool Calls:
  check_inbox (call_bc4a575d129a423f891bdf73)
 Call ID: call_bc4a575d129a423f891bdf73
  Args:
================================= Tool Message =================================
Name: check_inbox

[{"subject": "周末见个面？", "content": "\n                嗨 虎哥，\n                我下周会去城里，不知道我们有没有机会一起喝杯咖啡？\n\n                祝好，简\n            ", "from": "jane@itcast.cn", "status": "unread"}, {"subject": "周五会议", "content": "\n                嗨 虎哥，\n                非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？\n\n                祝好，小李\n            ", "from": "lixiaolong@itcast.cn", "status": "checked"}]
================================== Ai Message ==================================

您的收件箱中有 2 封邮件：

**1. 周末见个面？** (未读)
- 发件人：jane@itcast.cn
- 内容：简邀请您下周一起喝咖啡

**2. 周五会议** (已查看)
- 发件人：lixiaolong@itcast.cn
- 内容：小李无法准时参加周五的会议，请求重新安排时间

需要我帮您回

## Customized Middleware
- Node-Style
    - before/after model
    - before/after tool
    - before/after agent
- Wrap-Style
    - wrap model call
    - wrap tool call

In [19]:
from langgraph.runtime import Runtime
from langchain.agents import AgentState
from langchain.agents.middleware import after_model


# 1.定义自定义state，记录模型调用次数
class CustomState(AgentState):
    """扩展Agent状态，添加自定义字段"""
    model_call_count: int  # 模型调用次数


# 2.定义中间件
@after_model(state_schema=CustomState)
def increment_counter(
    state: CustomState,
    runtime: Runtime
):

    # 读取State中的count值
    current_count = state.get("model_call_count", 0)
    # 返回dict，其中的key/value是要更新的State值
    return {"model_call_count": current_count + 1}

In [20]:
# 创建智能体，设置middleware
agent = create_agent(
    model=qwen_model,
    middleware=[increment_counter],
    checkpointer=InMemorySaver()
)
config = {"configurable": {"thread_id": "1"}}

# 调用智能体
response = agent.invoke(
    {"messages": [HumanMessage("Hello")],},
    config
)

print(response)

{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='297a5e4a-f53e-46ee-9218-8e85c1eac9de'), AIMessage(content='Hello! How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 195, 'prompt_tokens': 11, 'total_tokens': 206, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 183, 'rejected_prediction_tokens': None, 'text_tokens': 195}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'text_tokens': 11}}, 'model_provider': 'openai', 'model_name': 'qwen3.7-plus', 'system_fingerprint': None, 'id': 'chatcmpl-5f960cd7-c58c-9e07-ba01-de7ed4442138', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a10075-f5f1-7732-963b-ca965b0bdf81-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 195, 'total_tokens': 206, 'input_token_details': {'cache_r

In [21]:
from langchain.agents.middleware import (
    wrap_model_call,
    ModelRequest,
    ModelResponse,
)
from typing import Any, Callable


@wrap_model_call
def retry_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    for attempt in range(3):
        try:
            return handler(request) # 这一步就是调用模型
        except Exception as e:
            print(f"Retry {attempt + 1}/3 after error: {e}")
            if attempt == 2:
                raise
    return None


agent = create_agent(
    model=qwen_model,
    middleware=[retry_model]
)

config = {"configurable": {"thread_id": "1"}}

# 调用智能体，并初始化state
response = agent.invoke(
    {"messages": [HumanMessage("Hello")]},
    config,
)

print(response)


{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='f3480f29-131b-4cbd-b05a-c3cdbba8d94b'), AIMessage(content='Hello! How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 227, 'prompt_tokens': 11, 'total_tokens': 238, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 215, 'rejected_prediction_tokens': None, 'text_tokens': 227}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'text_tokens': 11}}, 'model_provider': 'openai', 'model_name': 'qwen3.7-plus', 'system_fingerprint': None, 'id': 'chatcmpl-f5461895-ee24-9c47-b260-84624bad990d', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a10077-c404-7881-a092-f5b8b4aec539-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 227, 'total_tokens': 238, 'input_token_details': {'cache_r

In [24]:
from langchain.agents.middleware import AgentMiddleware
from langchain.agents.middleware.types import ModelCallResult, ToolCallRequest
from langchain_core.messages.tool import ToolMessage
from langgraph.types import Command


class LoggingMiddleware(AgentMiddleware):

    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelCallResult:
        try:
            print(f"\n=======About to call model with {len(request.messages)} messages=======")
            return handler(request)
        except Exception as e:
            print(f"\n=======[错误]: {str(e)}=======")
            return AIMessage("调用模型失败，请重试~")

    def wrap_tool_call(
        self,
        request: ToolCallRequest,
        handler: Callable[[ToolCallRequest], ToolMessage | Command],
    ) -> ToolMessage | Command:
        print(f"\n=======调用工具: {request.tool_call['name']}=======")
        print(f"\n=======参数: {request.tool_call['args']}=======")
        try:
            result = handler(request)
            print("\n=======工具调用成功！=======")
            return result
        except Exception as e:
            print(f"\n=======工具调用失败: {e}=======")
            raise


@tool
def get_weather(location: str):
    """查询指定城市的天气信息"""
    return f"Current weather in {location} is sunny, 25℃."


agent = create_agent(
    model=qwen_model,
    middleware=[LoggingMiddleware()],
    tools=[get_weather],
)

for chunk, metadata in agent.stream(
    {"messages": [HumanMessage("杭州今天天气如何？")]},
    stream_mode="messages"
):
    if chunk and chunk.content:
        print(chunk.content, end="", flush=True)


=======About to call model with 1 messages=======

=======调用工具: get_weather=======

=======参数: {'location': '杭州'}=======

=======工具调用成功！=======
Current weather in 杭州 is sunny, 25℃.
=======About to call model with 3 messages=======
杭州今天天气晴朗，气温为25℃。

In [28]:
from langchain.agents.middleware import wrap_model_call
from collections.abc import Callable
from pydantic.dataclasses import dataclass

# 定义一个context，记录用户是否开启思考模式
@dataclass
class UserContext:
    thinking: bool = False


@wrap_model_call
def dynamic_model_settings(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    # 获取上下文参数thinking，是否开启思考模式
    thinking = request.runtime.context.thinking

    # 如果Thinking为True，则开启思考模式，否则关闭思考模式
    enable_thinking = True if thinking else False
    model_settings = {"extra_body": {"enable_thinking": enable_thinking}}

    print(request.model_settings)

    # 覆盖request中的model参数
    modified_request = request.override(model_settings=model_settings)
    return handler(modified_request)

# 创建智能体，设置middleware
agent = create_agent(
    model=qwen_model,
    middleware=[dynamic_model_settings],
    context_schema=UserContext
)

# 调用智能体，并初始化state
response = agent.invoke(
    {"messages": [HumanMessage("How are you? Are you handsome?")]},
    context=UserContext(thinking=False) # 通过Context控制思考模式开关
)

print(response)

{}
{'messages': [HumanMessage(content='How are you? Are you handsome?', additional_kwargs={}, response_metadata={}, id='8b94d5c5-87e3-4b17-a675-0e4ae7c6224f'), AIMessage(content="I'm doing well, thank you for asking! I hope you are having a great day too.\n\nAs for being handsome—since I don’t have a physical body, face, or style, I can’t exactly pull off a good look in the mirror! 😄 However, I do strive to be **elegant** in my code and **polished** in my responses. If clean logic and helpful answers count as good looks, then I’d like to think I’m doing pretty well!\n\nHow can I help you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 109, 'prompt_tokens': 20, 'total_tokens': 129, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'text_tokens': 20}}, 'model_provider': 'openai', 'model_name': 'qwen3.7-plus', 'system_fingerprint': None, 'id': 'chatcmpl-2d

In [32]:
from langgraph.runtime import Runtime
from typing import NotRequired, Any
from langchain_core.messages import AIMessage, HumanMessage
from langchain.agents.middleware import AgentMiddleware, hook_config, AgentState


# 1.定义自定义state，记录模型调用次数
class CustomAgentState(AgentState):
    """扩展Agent状态，添加自定义字段"""
    model_call_count: NotRequired[int]  # 模型调用次数


# 2.定义中间件
class ModelCallLimitMiddleware(AgentMiddleware[CustomAgentState]):

    def __init__(self, max_limit: int = 10):
        super().__init__()
        self.max_limit = max_limit

    # after_model hook，模型调用次数计数
    def after_model(self, state: CustomAgentState, runtime: Runtime) -> dict[str, Any] | None:
        current_count = state.get("model_call_count", 0)
        return {"model_call_count": current_count + 1}

    # before_model hook，校验模型调用次数是否超限
    @hook_config(can_jump_to=["end"])
    def before_model(self, state: CustomAgentState, runtime: Runtime) -> dict[str, Any] | None:
        # 获取模型调用次数
        current_count = state.get("model_call_count", 0) + 1
        # 判断是否超限
        print(f"当前模型调用次数：{current_count}/{self.max_limit}")
        if current_count > self.max_limit:
            return {
                "jump_to": "end",
                "messages": AIMessage(f"模型调用次数超过最大限制：{self.max_limit}！")
            }
        return None

# 创建智能体，设置middleware
agent = create_agent(
    model=qwen_model,
    middleware=[ModelCallLimitMiddleware(max_limit=0)],
    checkpointer=InMemorySaver(),
    state_schema=CustomAgentState
)
config = {"configurable": {"thread_id": "1"}}

# 调用智能体，并初始化state
response = agent.invoke({
    "messages": [HumanMessage("Hello")],
    "model_call_count": 0,
}, config)

print(response)

当前模型调用次数：1/0
{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='15cd3447-8558-4a15-92b6-8c48f06b9e2e'), AIMessage(content='模型调用次数超过最大限制：0！', additional_kwargs={}, response_metadata={}, id='a7f70b42-f48e-4e49-96cc-50376ee5974b', tool_calls=[], invalid_tool_calls=[])], 'model_call_count': 0}
